# 非線形バネへのC/GMRESの適用

アルゴリズム理解のため非線形バネにたいしてC/GMRESを適用する。

## 1質点モデル

<img src ="images/one-degree-nonlinear-spring.png" style="width:30%;"/>

### 運動方程式

バネの復元力 $F(x)$ は次のように非線形性を持つとする。

$$
F(x) = kx(t) + k_3 x(t)^3
$$

$x(t)$ の方向に力 $u(t)$ を質点 $m$ へ加えた場合、次の運動方程式になる。

$$
m\ddot{x}(t) = u(t) - k(x) - k_3 x(t)^3
$$

### 状態方程式

状態 $X(t)$ を以下のように定義する。

$$
X(t) = [x(t), v(t)]^T
$$

運動方程式を以下のようにまとめる。

$$
\begin{aligned}
\dot{x}(t) &= v(t) \\
\dot{v}(t) &= \frac{1}{m}\left( u(t) - k x(t) - k_3 x(t)^3 \right) 
\end{aligned}
$$

ここから、状態方程式 $\dot{X} = f(X, u)$ は次のようになる。

$$
\frac{d}{dt}
\begin{bmatrix}
x(t) \\ v(t)
\end{bmatrix}
= \begin{bmatrix}
v(t) \\ \dfrac{1}{m}\left( u(t) - k x(t) - k_3 x(t)^3 \right)
\end{bmatrix}
$$

### 拡大評価関数とコスト

拡大評価関数 $\bar{J}$ は次の式である。 

$$
\bar{J} = \Phi(X(T)) + \int_0^T \left[ L(X,u) + \lambda^T \ \left( f(X,u) - \dot{X} \right) \right] dt
$$

位置の目標値を$x_{ref}$ とするように、ランニングコスト$L(X,u)$と終端コスト$\Phi(X(T))$ を設定する。


##### ランニングコスト $L$

ランニングコスト $L(X,u)$ は次のように設定する。

$$
L(X,u) = \frac{1}{2} q_x(x(t) - x_{ref})^2 + \frac{1}{2} q_v v(t)^2 + \frac{1}{2} r u(t)^2
$$

$\bar{J}$の最小化問題になるため、ランニングコスト $L(X,u)$ のそれぞれの意味は次のようになる。

- 第1項目: 位置の目標値との誤差を小さくする
- 第2項目: 速度を大きくしない
- 第3項目: 入力を大きくしない

##### 終端コスト $\Phi$

終端コスト $\Phi(X(T))$ は次のように設定する。

$$
\Phi(X(T)) = \frac{1}{2}q_{xT}(x(T) - x_{ref})^2 + \frac{1}{2}q_{vT}v(T)^2
$$

$\bar{J}$の最小化問題になるため、終端コスト $\Phi(X(T))$ のそれぞれの意味は次のようになる。

- 第1項目: 時刻$T$ における位置と目標位置との誤差を小さくする
- 第2項目: 時刻$T$ における速度を大きくしない

##### 随伴変数　$\lambda$

$\lambda$は状態方程式制約$f(X,u) - \dot{X}$ と内積をとり、$\lambda^T (f(X,u) - \dot{X}) $がスカラーとなるように設定する。状態$X$は2変数であるため、$\lambda$を次のように設定する。

$$
\lambda = [\lambda_x, \lambda_v]^T
$$


### PMP 条件

Hamiltonian $H$ は次の式である。

$$
H(X,u,\lambda) = L(X, u) + \lambda^T f(X, u)
$$

具体的に$H$を構成すると次のようになる。

$$
H = \frac{1}{2} q_x(x(t) - x_{ref})^2 + \frac{1}{2} q_v v(t)^2 + \frac{1}{2} r u(t)^2 + \lambda_x v(t) + \frac{\lambda_v}{m}\left(u(t) - k x(t) - k_3 x(t)^3  \right)
$$

PMP条件を構成するため、$H$ を$X,u$で偏微分、$\Phi$を$X(T)$ で偏微分を行う。また、$H_\lambda = f(X,u)$ である。

$$
\begin{aligned}
H_X &= \frac{\partial H}{\partial X} = \begin{bmatrix} \partial H / \partial x \\ \partial H / \partial v \end{bmatrix} =
\begin{bmatrix}
q_x(x(t) - x_{ref}) - \lambda_v/m \ \left( k + 3 k_3 x(t)^2 \right) \\
q_v v(t) + \lambda_x 
\end{bmatrix} \\
H_u &= \frac{\partial H}{\partial u} = r u(t) + \frac{\lambda_v}{m} \\
\Phi_X &= \frac{\partial \Phi}{\partial X} = \begin{bmatrix} \partial \Phi / \partial x \\ \partial \Phi / \partial v \end{bmatrix} =
\begin{bmatrix}
q_{xT}(x(T) - x_{ref}) \\
q_{vT} v(T)
\end{bmatrix} 
\end{aligned}
$$



$H$を用いて、PMP条件は次のようになる。

$$
\begin{array}{l}
\dot{X} = H_\lambda \\
\dot{\lambda} = - H_X \\
H_u = 0 \\
\lambda(T) = \Phi_X(X(T)) \\
\end{array}
$$ 

よって1質点の非線形バネモデルにおけるPMP条件の式は以下となる。

状態方程式

$$
\boxed{
\frac{d}{dt}
\begin{bmatrix}
x(t) \\ v(t)
\end{bmatrix}
= \begin{bmatrix}
v(t) \\ \dfrac{1}{m}\left( u(t) - k x(t) - k_3 x(t)^3 \right)
\end{bmatrix}
}
$$

随伴方程式

$$
\boxed{
\frac{d}{dt}
\begin{bmatrix}
\lambda_x \\ \lambda_v
\end{bmatrix}
=-\begin{bmatrix}
q_x(x(t) - x_{ref}) - \lambda_v/m \ \left( k + 3 k_3 x(t)^2 \right) \\
q_v v(t) + \lambda_x 
\end{bmatrix}
}
$$

停留条件

$$
\boxed{
r u(t) + \frac{\lambda_v}{m} = 0
}
$$

終端条件

$$
\boxed{
\begin{bmatrix}
\lambda_x(T) \\ \lambda_v(T)
\end{bmatrix} =
\begin{bmatrix}
q_{xT}(x(T) - x_{ref}) \\
q_{vT} v(T)
\end{bmatrix} 
}
$$

### 状態 $X$ 随伴変数 $\lambda$ の時系列の計算式と停留条件式 $F$ の構成

制御入力列 $U(t_k)$ 、状態の初期値 $X[0]$、予測ホライゾンの長さ$T(t_k)=T_f(1 - \exp(-\alpha t_k))$ の$t_k$を設定値として時系列を計算する。

予測ホライゾンのステップ幅 $h(t_k)$ は $T(t_k)$の分割数 $N$ より以下のようになる。

$$
h(t_k) = T(t_k) / N
$$

以下の式は上から順に、状態$X[n]$、終端状態 $\lambda[N]$、随伴変数 $\lambda[n]$ に関する。

$$
\begin{aligned}
\begin{bmatrix}
x[n+1] \\ v[n+1]
\end{bmatrix} &=
\begin{bmatrix}
x[n] \\ v[n]
\end{bmatrix} +
h\begin{bmatrix}
v[n] \\ \dfrac{1}{m}\left( u_n(t_k) - k x[n] - k_3 x[n]^3 \right)
\end{bmatrix}, \quad n = 0, \cdots , N-1 \\
\begin{bmatrix}
\lambda_x[N] \\ \lambda_v[N]
\end{bmatrix} &=
\begin{bmatrix}
q_{xT}(x[N] - x_{ref}) \\
q_{vT} v[N]
\end{bmatrix} \\
\begin{bmatrix}
\lambda_x[n] \\ \lambda_v[n]
\end{bmatrix} &=\begin{bmatrix}
\lambda_x[n+1] \\ \lambda_v[n+1]
\end{bmatrix}
+h\begin{bmatrix}
q_x(x[n] - x_{ref}) - \lambda_v[n+1]/m \ \left( k + 3 k_3 x[n]^2 \right) \\
q_v v[n] + \lambda_x[n+1] 
\end{bmatrix} , \quad n = N-1, \cdots , 1
\end{aligned}
$$

上記を用いて、停留条件式 $F(X(t_k), U(t_k))$ を以下のように構成する。

$$
F(X(t_k), U(t_k)) = 
\begin{bmatrix}
r u_0(t) + \dfrac{\lambda_v[1]}{m} \\
\vdots \\
r u_{N-1}(t) + \dfrac{\lambda_v[N]}{m} \\
\end{bmatrix} = 0
$$

ここまでがC/GMRESを計算するための設定である。

また、この時系列から$F$を構成する計算はC/GMRESの実行中に何度も行われる。そのため

$$
F(U(t_k), X(t_k), t_k)
$$

を関数として定義する。これは、$U(t_k)$ に制御入力列、$X(t_k)$ に実時間 $t_k$ における現在状態を与え、それを予測区間の初期状態$X[0]$として使用する、$t_k$を$T(t_k)$ を計算する入力として与えれれば、それをもとに停留条件の式 $F(U(t_k), X(t_k), t_k)$ を構成する、という意味である。

# C/GMRESの計算

## 初期制御入力 $u(0)$ の計算

$T(0)=0$ として解析的に $u(0)$ を求める。

停留条件を$T(0)=0$ で構成すると以下の式となる。

$$
r u(0) + \frac{q_{vT} v(0)}{m} = 0 \rightarrow u_0 = -\frac{q_{vT}v(0)}{m} 
$$

ここから、初期状態 $X(0)$ を取得すれば、$u(0)$が計算できるといえる。
